In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns

from scipy.optimize import fsolve


# Data:

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from typing import Dict, Optional, Tuple, Any, List
import geopandas as gpd 
from multiprocessing import Pool, cpu_count
import warnings
from tqdm import tqdm
import logging
from functools import partial

# Ignore specific RuntimeWarnings for numerical stability
warnings.filterwarnings('ignore', category=RuntimeWarning, message='invalid value encountered in true_divide')
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
np.seterr(all="ignore")

# --- GLOBAL CONSTANTS & PATHS ---
PROJECT_ROOT = r"C:\Users\hdagne1\Box\Dr.Mesfin Research\Codes\DA\DA_Github_repo\Bayesian_DA_Budyko_modeling"
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
INPUT_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "input_data", "NDVI") # New directory for M and Slope
sys.path.append(PROJECT_ROOT)
G_MAX_CEILING = 5000.0

try:
    from src.model import ModelParams, two_store_model_step
    from src.enkf import EnKFConfig, enkf_update, enkf_forecast_step
    from src.param_manager import get_calibrated_params_for_basin, load_all_calibrated_params
    from src.metrics import calculate_kge
    # from src.budyko import estimate_budyko_et, OmegaMLRModel
except ImportError as e:
    print(f"FATAL: CORE MODULE Import failed. Using Fallbacks. Error: {e}")
    


def load_feather_df(fname: str, ddir: str) -> pd.DataFrame:
    """Loads a Feather file from a specified directory."""
    path = os.path.join(ddir, fname)
    if not os.path.exists(path):
        # Allow Budyko files to fail gracefully if the input_data directory is wrong, but log it
        if ddir == INPUT_DATA_DIR:
             logging.error(f"FATAL: Budyko data file not found: {path}. Check your path!")
        else:
            logging.warning(f"File not found: {path}. Returning empty DataFrame.")
        return pd.DataFrame()
    df = pd.read_feather(path)
    df = df.dropna(axis=1, how='all')
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        df.set_index('time', inplace=True)
    return df


Rainf_df = load_feather_df("Rainf.feather", DATA_DIR)
PotEvap_df = load_feather_df("PotEvap.feather", DATA_DIR)
Evap_df = load_feather_df("EVap.feather", DATA_DIR)
Q_USGS_monthly = load_feather_df("Q_USGS.feather", DATA_DIR)
Qsb_monthly = load_feather_df("Qsb.feather", DATA_DIR)
Q_nldas_mm_monthly = load_feather_df("Q_nldas_mm_monthly.feather", DATA_DIR)
S_init_df = load_feather_df("RootMoist.feather", DATA_DIR)
G_init_df = load_feather_df("SoilM_0_200cm.feather", DATA_DIR)

# Budyko Parameter DataFrames (from INPUT_DATA_DIR)
M_df = load_feather_df("M.feather", INPUT_DATA_DIR)
Slope_basin = load_feather_df("slope.feather", INPUT_DATA_DIR) # Single-row DataFrame
M_basin = M_df[Slope_basin.columns]
M_basin.index = pd.to_datetime(M_basin.index, format='%Y-%m')
# M_basin.index = pd.to_datetime(M_basin.index + '-01')
M_basin = M_basin.loc[Evap_df.index]



In [7]:
dataframes = {
    "Rainf_df": Rainf_df,
    "PotEvap_df": PotEvap_df,
    "Evap_df": Evap_df,
    "Q_USGS_monthly": Q_USGS_monthly,
    "Qsb_monthly": Qsb_monthly,
    "Q_nldas_mm_monthly": Q_nldas_mm_monthly,
    "S_init_df": S_init_df,
    "G_init_df": G_init_df,
    "M_df": M_df,
    "Slope_basin": Slope_basin,
    "M_basin": M_basin
}

for name, df in dataframes.items():
    print(f"{name}: {df.shape}")


Rainf_df: (180, 505)
PotEvap_df: (180, 505)
Evap_df: (180, 505)
Q_USGS_monthly: (180, 607)
Qsb_monthly: (180, 505)
Q_nldas_mm_monthly: (180, 505)
S_init_df: (180, 505)
G_init_df: (180, 505)
M_df: (228, 671)
Slope_basin: (1, 505)
M_basin: (180, 505)


In [ ]:


def solve_for_omega(ET, QB, PET, omega_guess=1.0):
    ratio_ET = ET / (ET + QB)
    ratio_PET = PET / (ET + QB)

    def f(omega):
        # equation = LHS - RHS
        return 1 + ratio_PET - (1 + ratio_PET**omega)**(1/omega) - ratio_ET

    try:
        sol = fsolve(f, x0=omega_guess, xtol=1e-8)
        # fsolve returns array
        return sol[0] if np.isfinite(sol[0]) else np.nan
    except:
        return np.nan

omega_true = pd.DataFrame(index=Evap_df.index, columns=Evap_df.columns, dtype=float)

for col in tqdm(Evap_df.columns, desc="Solving omega for basins"):
    ET_col = Evap_df[col].values
    QB_col = Qsb_monthly[col].values
    PET_col = PotEvap_df[col].values
    ET_Ke = 0.68 * PET_col   # ET_ke = ke * PET
    omega_values = np.array([solve_for_omega(ET, QB, PET) 
                             for ET, QB, PET in zip(ET_Ke, QB_col, PET_col)])
    
    omega_true[col] = omega_values


# Omega true:

In [ ]:
def solve_for_omega(ET, QB, PET, omega_guess=1.0):
    ratio_ET = ET / (ET + QB)
    ratio_PET = PET / (ET + QB)

    def f(omega):
        return 1 + ratio_PET - (1 + ratio_PET**omega)**(1/omega) - ratio_ET

    try:
        sol = fsolve(f, x0=omega_guess, xtol=1e-8)
        return sol[0] if np.isfinite(sol[0]) else np.nan
    except:
        return np.nan

def compute_omega(Evap_df: pd.DataFrame, Qsb_monthly: pd.DataFrame, PotEvap_df: pd.DataFrame, ke=0.68) -> pd.DataFrame:
    """
    Compute omega for basins given Evaporation, Baseflow, and Potential Evaporation dataframes.

    Parameters:
        Evap_df: pd.DataFrame - Actual evapotranspiration data.
        Qsb_monthly: pd.DataFrame - Baseflow monthly data.
        PotEvap_df: pd.DataFrame - Potential evapotranspiration data.
        ke: float - Multiplicative factor for PET to estimate ET (default 0.68).

    Returns:
        pd.DataFrame of omega values with the same shape as input dataframes.
    """
    omega_true = pd.DataFrame(index=Evap_df.index, columns=Evap_df.columns, dtype=float)

    for col in tqdm(PotEvap_df.columns, desc="Solving omega for basins"):
        ET_col = Evap_df[col].values
        QB_col = Qsb_monthly[col].values
        PET_col = PotEvap_df[col].values
        ET_Ke = ke * PET_col
        omega_values = np.array([solve_for_omega(ET, QB, PET)  for ET, QB, PET in zip(ET_Ke, QB_col, PET_col)])
        omega_true[col] = omega_values

    return omega_true
omega_true = compute_omega(Evap_df, Qsb_monthly, PotEvap_df)
omega_true

In [ ]:
Q_nldas_mm_monthly

# omega MLR:

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass



def solve_for_omega(ET, QB, PET, omega_guess=1.0):
    ratio_ET = ET / (ET + QB)
    ratio_PET = PET / (ET + QB)

    def f(omega):
        return 1 + ratio_PET - (1 + ratio_PET**omega)**(1/omega) - ratio_ET

    try:
        sol = fsolve(f, x0=omega_guess, xtol=1e-8)
        return sol[0] if np.isfinite(sol[0]) else np.nan
    except:
        return np.nan

def compute_omega(Evap_df: pd.DataFrame, Qsb_monthly: pd.DataFrame, PotEvap_df: pd.DataFrame, ke=0.68) -> pd.DataFrame:
    """
    Compute omega for basins given Evaporation, Baseflow, and Potential Evaporation dataframes.

    Parameters:
        Evap_df: pd.DataFrame - Actual evapotranspiration data.
        Qsb_monthly: pd.DataFrame - Baseflow monthly data.
        PotEvap_df: pd.DataFrame - Potential evapotranspiration data.
        ke: float - Multiplicative factor for PET to estimate ET (default 0.68).

    Returns:
        pd.DataFrame of omega values with the same shape as input dataframes.
    """
    omega_true = pd.DataFrame(index=Evap_df.index, columns=Evap_df.columns, dtype=float)

    for col in tqdm(PotEvap_df.columns, desc="Solving omega for basins"):
        ET_col = Evap_df[col].values
        QB_col = Qsb_monthly[col].values
        PET_col = PotEvap_df[col].values
        ET_Ke = ke * PET_col
        omega_values = np.array([solve_for_omega(ET, QB, PET)  for ET, QB, PET in zip(ET_Ke, QB_col, PET_col)])
        omega_true[col] = omega_values

    return omega_true
omega_true = compute_omega(Evap_df, Qsb_monthly, PotEvap_df)
omega_true





@dataclass
class OmegaMLRModel:
    beta0: float
    beta1: float
    beta2: float

    def predict(self, M: np.ndarray, Slope: np.ndarray) -> np.ndarray:
        """Predict omega from M and Slope."""
        omega_MLR = self.beta0 + self.beta1 * M + self.beta2 * Slope
        return np.clip(omega_MLR, 1.0, 10.0)


def fit_omega_mlr(M: np.ndarray, Slope: np.ndarray, omega_true: np.ndarray) -> np.ndarray:
    """
    Fit omega_true to M and Slope for each basin using MLR.
    Returns omega_fitted with same shape as M (time × basins).
    """
    # # Convert to NumPy arrays if they are DataFrames
    # M = np.asarray(M)
    # omega_true = np.asarray(omega_true)
    # Slope = np.asarray(Slope)

    # Flatten Slope to 1D if shape is (1, basins)
    if Slope.ndim == 2 and Slope.shape[0] == 1:
        Slope_flat = Slope.flatten()
    elif Slope.ndim == 1:
        Slope_flat = Slope
    else:
        raise ValueError(f"Slope shape not recognized: {Slope.shape}")

    n_time, n_basins = M.shape
    omega_fitted = np.full_like(M, np.nan, dtype=float)

    for i in range(n_basins):
        M_col = M[:, i]
        Slope_val = Slope_flat[i]
        Slope_col = np.full(n_time, Slope_val)

        valid_idx = ~(np.isnan(M_col) | np.isnan(Slope_col) | np.isnan(omega_true[:, i]))
        if valid_idx.sum() < 3:
            omega_fitted[:, i] = 2.36 + 1.16 * M_col
            continue

        X = np.vstack([np.ones(valid_idx.sum()), M_col[valid_idx], Slope_col[valid_idx]]).T
        y = omega_true[valid_idx, i]

        try:
            coef, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
            if len(coef) < 3 or np.any(np.isnan(coef)):
                X_M = np.vstack([np.ones(valid_idx.sum()), M_col[valid_idx]]).T
                coef_M, _, _, _ = np.linalg.lstsq(X_M, y, rcond=None)
                omega_fitted[:, i] = coef_M[0] + coef_M[1] * M_col
            else:
                omega_fitted[:, i] = coef[0] + coef[1] * M_col + coef[2] * Slope_col
        except np.linalg.LinAlgError:
            omega_fitted[:, i] = 2.36 + 1.16 * M_col

    return np.clip(omega_fitted, 1.0, 10.0)



M_array = M_basin.values if hasattr(M_basin, "values") else M_basin
Slope_array = Slope_basin.values if hasattr(Slope_basin, "values") else Slope_basin
omega_true_array = omega_true.values if hasattr(omega_true, "values") else omega_true

# Fit omega
omega_fitted_array = fit_omega_mlr(M_array, Slope_array, omega_true_array)

# Convert to DataFrame with same index & columns as M_basin
omega_MLR = pd.DataFrame(omega_fitted_array, index=M_basin.index, columns=M_basin.columns)


def estimate_budyko_et_from_omega(ET: pd.DataFrame, Qb: pd.DataFrame, PET_df: pd.DataFrame, 
                                  omega_MLR: pd.DataFrame) -> pd.DataFrame:
    """
    Estimate ET using Fu Budyko model with dynamic omega from omega_MLR.
    
    Parameters:
    - ET: Actual ET DataFrame (mm), shape (time x basins) or can be initial ET estimate
    - Qb: Baseflow/discharge DataFrame (mm), same shape as ET
    - PET_df: Potential ET DataFrame (mm), same shape as ET
    - omega_MLR: Precomputed omega DataFrame (time x basins)
    
    Returns:
    - ET_df: Estimated ET (time x basins) DataFrame
    """
    # Combine ET and baseflow
    ET_Qb = ET + Qb

    # Compute aridity ratio safely
    aridity = np.divide(PET_df.values, ET_Qb.values, out=np.zeros_like(PET_df.values), where=ET_Qb.values>0)

    # Fu Budyko formula: E/(ET+Qb) = 1 + φ - (1 + φ^ω)^(1/ω)
    with np.errstate(over='ignore', invalid='ignore'):
        E_ratio = 1.0 + aridity - np.power(1.0 + np.power(aridity, omega_MLR.values), 1.0 / omega_MLR.values)
        E_ratio = np.nan_to_num(E_ratio, nan=0.0, posinf=0.0, neginf=0.0)

    # Estimated ET
    ET_est = ET_Qb.values * E_ratio

    # Clip ET: cannot exceed PET or ET + Qb
    ET_est = np.clip(ET_est, 0.0, np.minimum(PET_df.values, ET_Qb.values * 0.999))

    # Return as DataFrame
    ET_B = pd.DataFrame(ET_est, index=PET_df.index, columns=PET_df.columns)
    return ET_B

ET_B = estimate_budyko_et_from_omega(ET=Evap_df, Qb=Qsb_monthly, PET_df=PotEvap_df, omega_MLR=omega_MLR)
ET_B


In [20]:
import sys
import os
import numpy as np
import pandas as pd
from typing import Dict, Optional, Tuple, Any, List
import geopandas as gpd 
from multiprocessing import Pool, cpu_count
import warnings
from tqdm import tqdm
import logging
from functools import partial

# Ignore specific RuntimeWarnings for numerical stability
warnings.filterwarnings('ignore', category=RuntimeWarning, message='invalid value encountered in true_divide')
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
np.seterr(all="ignore")

# --- GLOBAL CONSTANTS & PATHS ---
PROJECT_ROOT = r"C:\Users\hdagne1\Box\Dr.Mesfin Research\Codes\DA\DA_Github_repo\Bayesian_DA_Budyko_modeling"
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
INPUT_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "input_data", "NDVI") # New directory for M and Slope
sys.path.append(PROJECT_ROOT)
G_MAX_CEILING = 5000.0

try:
    from src.model import ModelParams, two_store_model_step
    from src.enkf import EnKFConfig, enkf_update, enkf_forecast_step
    from src.param_manager import get_calibrated_params_for_basin, load_all_calibrated_params
    from src.metrics import calculate_kge
    # from src.budyko import estimate_budyko_et, OmegaMLRModel
except ImportError as e:
    print(f"FATAL: CORE MODULE Import failed. Using Fallbacks. Error: {e}")
    


def load_feather_df(fname: str, ddir: str) -> pd.DataFrame:
    """Loads a Feather file from a specified directory."""
    path = os.path.join(ddir, fname)
    if not os.path.exists(path):
        # Allow Budyko files to fail gracefully if the input_data directory is wrong, but log it
        if ddir == INPUT_DATA_DIR:
             logging.error(f"FATAL: Budyko data file not found: {path}. Check your path!")
        else:
            logging.warning(f"File not found: {path}. Returning empty DataFrame.")
        return pd.DataFrame()
    df = pd.read_feather(path)
    df = df.dropna(axis=1, how='all')
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        df.set_index('time', inplace=True)
    return df

Rainf_df = load_feather_df("Rainf.feather", DATA_DIR)
PotEvap_df = load_feather_df("PotEvap.feather", DATA_DIR)
Evap_df = load_feather_df("EVap.feather", DATA_DIR)
Q_USGS_monthly = load_feather_df("Q_USGS.feather", DATA_DIR)
Qsb_monthly = load_feather_df("Qsb.feather", DATA_DIR)
Q_nldas_mm_monthly = load_feather_df("Q_nldas_mm_monthly.feather", DATA_DIR)
S_init_df = load_feather_df("RootMoist.feather", DATA_DIR)
G_init_df = load_feather_df("SoilM_0_200cm.feather", DATA_DIR)

# Budyko Parameter DataFrames (from INPUT_DATA_DIR)
M_df = load_feather_df("M.feather", INPUT_DATA_DIR)
Slope_df = load_feather_df("slope.feather", INPUT_DATA_DIR) # Single-row DataFrame


Rainf_df = load_feather_df("Rainf.feather", DATA_DIR)
PotEvap_df = load_feather_df("PotEvap.feather", DATA_DIR)
Evap_df = load_feather_df("EVap.feather", DATA_DIR)
Q_USGS_monthly = load_feather_df("Q_USGS.feather", DATA_DIR)
Qsb_monthly = load_feather_df("Qsb.feather", DATA_DIR)
Q_nldas_mm_monthly = load_feather_df("Q_nldas_mm_monthly.feather", DATA_DIR)
S_init_df = load_feather_df("RootMoist.feather", DATA_DIR)
G_init_df = load_feather_df("SoilM_0_200cm.feather", DATA_DIR)

# Budyko Parameter DataFrames (from INPUT_DATA_DIR)
M_df = load_feather_df("M.feather", INPUT_DATA_DIR)
Slope_basin = load_feather_df("slope.feather", INPUT_DATA_DIR) # Single-row DataFrame
M_basin = M_df[Slope_df.columns]
M_basin.index = pd.to_datetime(M_basin.index, format='%Y-%m')
# M_basin.index = pd.to_datetime(M_basin.index + '-01')
M_basin = M_basin.loc[Evap_df.index]

M_basin

# Ignore specific RuntimeWarnings for numerical stability
warnings.filterwarnings('ignore', category=RuntimeWarning, message='invalid value encountered in true_divide')
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
np.seterr(all="ignore")

# --- GLOBAL CONSTANTS & PATHS ---
PROJECT_ROOT = r"C:\Users\hdagne1\Box\Dr.Mesfin Research\Codes\DA\DA_Github_repo\Bayesian_DA_Budyko_modeling"
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
INPUT_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "input_data", "NDVI") # New directory for M and Slope
sys.path.append(PROJECT_ROOT)
G_MAX_CEILING = 5000.0

try:
    from src.model import ModelParams, two_store_model_step
    from src.enkf import EnKFConfig, enkf_update, enkf_forecast_step
    from src.param_manager import get_calibrated_params_for_basin, load_all_calibrated_params
    from src.metrics import calculate_kge
    from src.budyko import estimate_budyko_et, OmegaMLRModel
except ImportError as e:
    print(f"FATAL: CORE MODULE Import failed. Using Fallbacks. Error: {e}")
    


def load_feather_df(fname: str, ddir: str) -> pd.DataFrame:
    """Loads a Feather file from a specified directory."""
    path = os.path.join(ddir, fname)
    if not os.path.exists(path):
        # Allow Budyko files to fail gracefully if the input_data directory is wrong, but log it
        if ddir == INPUT_DATA_DIR:
             logging.error(f"FATAL: Budyko data file not found: {path}. Check your path!")
        else:
            logging.warning(f"File not found: {path}. Returning empty DataFrame.")
        return pd.DataFrame()
    df = pd.read_feather(path)
    df = df.dropna(axis=1, how='all')
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        df.set_index('time', inplace=True)
    return df

Rainf_df = load_feather_df("Rainf.feather", DATA_DIR)
PotEvap_df = load_feather_df("PotEvap.feather", DATA_DIR)
Evap_df = load_feather_df("EVap.feather", DATA_DIR)
Q_USGS_monthly = load_feather_df("Q_USGS.feather", DATA_DIR)
Qsb_monthly = load_feather_df("Qsb.feather", DATA_DIR)
Q_nldas_mm_monthly = load_feather_df("Q_nldas_mm_monthly.feather", DATA_DIR)
S_init_df = load_feather_df("RootMoist.feather", DATA_DIR)
G_init_df = load_feather_df("SoilM_0_200cm.feather", DATA_DIR)

# Budyko Parameter DataFrames (from INPUT_DATA_DIR)
M_df = load_feather_df("M.feather", INPUT_DATA_DIR)
Slope_df = load_feather_df("slope.feather", INPUT_DATA_DIR) # Single-row DataFrame



M_basin = M_df[Slope_df.columns]
M_basin.index = pd.to_datetime(M_basin.index, format='%Y-%m')
# M_basin.index = pd.to_datetime(M_basin.index + '-01')
M_basin = M_basin.loc[Evap_df.index]

M_basin

# Ignore specific RuntimeWarnings for numerical stability
warnings.filterwarnings('ignore', category=RuntimeWarning, message='invalid value encountered in true_divide')
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
np.seterr(all="ignore")

# --- GLOBAL CONSTANTS & PATHS ---
PROJECT_ROOT = r"C:\Users\hdagne1\Box\Dr.Mesfin Research\Codes\DA\DA_Github_repo\Bayesian_DA_Budyko_modeling"
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
INPUT_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "input_data", "NDVI") # New directory for M and Slope
sys.path.append(PROJECT_ROOT)
G_MAX_CEILING = 5000.0

try:
    from src.model import ModelParams, two_store_model_step
    from src.enkf import EnKFConfig, enkf_update, enkf_forecast_step
    from src.param_manager import get_calibrated_params_for_basin, load_all_calibrated_params
    from src.metrics import calculate_kge
    from src.budyko import BudykoModelEstimator, OmegaMLRModel
    
except ImportError as e:
    print(f"FATAL: CORE MODULE Import failed. Using Fallbacks. Error: {e}")

DATA_DIR = "./data_files/"
INPUT_DATA_DIR = "./input_data_files/"

def load_feather_df(filename, directory):
    """
    Placeholder for your custom feather loading function.
    
    Replace the body of this function with your actual implementation.
    Example: return pd.read_feather(os.path.join(directory, filename))
    """
    filepath = os.path.join(directory, filename)
    print(f"Loading: {filepath}")


    if filename in ["Rainf.feather", "PotEvap.feather", "EVap.feather", "Q_USGS.feather", 
                    "Qsb.feather", "Q_nldas_mm_monthly.feather", "RootMoist.feather", 
                    "SoilM_0_200cm.feather"]:
        n_time = 120
        n_basins = 5
        dates = pd.date_range(start='2000-01-01', periods=n_time, freq='M')
        basin_cols = [f'Basin_{i}' for i in range(1, n_basins + 1)]
        df = pd.DataFrame(np.random.rand(n_time, n_basins) * 50, index=dates, columns=basin_cols)
        return df
        
    elif filename == "M.feather":
        n_time = 125 # Longer to test the index slice
        n_basins = 5
        dates = pd.date_range(start='1999-08-01', periods=n_time, freq='M')
        basin_cols = [f'Basin_{i}' for i in range(1, n_basins + 1)]
        df = pd.DataFrame(np.random.rand(n_time, n_basins) * 2, index=dates, columns=basin_cols)
        return df
        
    elif filename == "slope.feather":
        n_basins = 5
        basin_cols = [f'Basin_{i}' for i in range(1, n_basins + 1)]
        df = pd.DataFrame(np.random.rand(1, n_basins) * 0.5 + 0.05, index=['Slope'], columns=basin_cols)
        return df
    # --- END DUMMY LOADING ---

# --- 2. Initialize and Run the Model Estimator ---
print("\n--- 2. Initializing and Running Budyko Model ---")

model = BudykoModelEstimator(
    Evap_df=Evap_df,
    Qsb_monthly=Qsb_monthly,
    PotEvap_df=PotEvap_df,
    M_basin=M_basin,
    Slope_basin=Slope_basin
)

# Run the full estimation process. The method returns the final ET_B.
ET_B = model.OmegaTrue_OmegaMLR_BudykoET()
ET_B

FATAL: CORE MODULE Import failed. Using Fallbacks. Error: cannot import name 'estimate_budyko_et' from 'src.budyko' (C:\Users\hdagne1\Box\Dr.Mesfin Research\Codes\DA\DA_Github_repo\Bayesian_DA_Budyko_modeling\src\budyko.py)

--- 2. Initializing and Running Budyko Model ---
Starting Budyko Model Estimation Workflow...
-> Step 1/3: Solving for omega_true...


Basins: 100%|██████████| 505/505 [00:12<00:00, 39.98it/s]


-> Step 2/3: Fitting MLR for dynamic omega...


Basins: 100%|██████████| 505/505 [00:00<00:00, 10618.01it/s]

-> Step 3/3: Estimating ET_B...
Workflow complete.


,06452000,13340000,06447000,06360500,06354000,05057000,07301500,06191500,02315500,06784000,...,12374250,10249300,01350140,14139800,02055100,13018300,01613050,09047700,02038850,09066200
time,,,,,,,,,,,,,,,,,,,,,
2000-01-01,0.013186,0.000000,0.012553,0.010241,0.008409,0.005005,0.069835,0.008170,0.098560,0.028078,...,-0.001579,0.048919,0.024644,0.000000,0.000000,0.000074,0.001678,0.010811,0.000000,0.011491
2000-02-01,0.030182,0.012188,0.026329,0.029190,0.024563,0.016692,0.116272,0.013404,0.145109,0.044883,...,0.001334,0.081249,0.004703,0.027070,0.083367,0.003226,0.040957,0.022469,0.095416,0.019823
2000-03-01,0.106284,0.050741,0.097695,0.103212,0.109406,0.108390,0.142475,0.030506,0.212526,0.119264,...,0.007886,0.152966,0.059531,0.083154,0.157636,0.007391,0.134454,0.028695,0.187318,0.027388
2000-04-01,0.162711,0.117281,0.161008,0.161729,0.170884,0.157076,0.209359,0.102157,0.236778,0.183121,...,0.093913,0.263676,0.102525,0.136989,0.165352,0.066537,0.147996,0.116482,0.191134,0.094866
2000-05-01,0.206303,0.170321,0.206345,0.226824,0.242608,0.224592,0.294205,0.229142,0.331988,0.219962,...,0.248378,0.340659,0.142707,0.165867,0.245454,0.289814,0.209658,0.271897,0.283441,0.205402
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-08-01,0.219292,0.199797,0.223519,0.219590,0.199054,0.220208,0.319744,0.176280,0.324287,0.206624,...,0.266024,0.302967,0.149262,0.196057,0.241902,0.116018,0.233621,0.166171,0.304075,0.145031
2014-09-01,0.162081,0.151095,0.156116,0.158708,0.156679,0.168868,0.221326,0.151457,0.202358,0.162759,...,0.194487,0.264073,0.127441,0.158305,0.189556,0.183106,0.199252,0.164704,0.238181,0.122822
2014-10-01,0.121948,0.094600,0.109994,0.110272,0.107161,0.115693,0.178106,0.086505,0.229829,0.135492,...,0.121797,0.174626,0.082584,0.086438,0.155984,0.121256,0.142252,0.119654,0.192649,0.079983


In [16]:
print("Evap_df shape:", Evap_df.shape)        # ET data (likely monthly or daily) for each basin
print("Qsb_monthly shape:", Qsb_monthly.shape)  # Baseflow or streamflow (monthly) for each basin
print("PotEvap_df shape:", PotEvap_df.shape)    # Potential evapotranspiration (same time frequency as Evap_df)
print("M_basin shape:", M_basin.shape)          # Basin metadata (static per basin, e.g., area, elevation)
print("Slope_basin shape:", Slope_basin.shape)  # Basin slope or topographic info (one value per basin)


Evap_df shape: (180, 505)
Qsb_monthly shape: (180, 505)
PotEvap_df shape: (180, 505)
M_basin shape: (180, 505)
Slope_basin shape: (1, 505)
